# Аугментации книг

Все шаги `book_transform` из `finetune/data_utils.py`. Первый кадр ряда — исходный кроп, дальше четыре независимых срабатывания.

Пропорции клейма не тянутся: отдельного растяжения нет, окно кропа квадратное (`ratio 1`). Гомография слабая, `distortion_scale 0.2`. В отдельных рядах шаги с вероятностью включены всегда, чтобы их было видно.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline
import torch
from PIL import Image
from torchvision import transforms

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "pipeline.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Запускайте ноутбук из репозитория CeramicNet")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from finetune.data_utils import book_transform, load_book_pil, load_books

torch.manual_seed(7)
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
configured = ROOT / "data" / "books" / "meta.csv"
if configured.is_file():
    books = load_books()
else:
    books = load_books(
        ROOT.parent / "artifacts_clean" / "meta.csv",
        ROOT.parent / "artifacts_clean" / "books",
    )
books = [sample for sample in books if sample.path.is_file()]
count = 16
if len(books) < count:
    raise RuntimeError(f"Нужно хотя бы {count} книжных кропов, найдено {len(books)}")

picked = [books[index * len(books) // count] for index in range(count)]
book_images = [(sample.mark_id, load_book_pil(sample.path)) for sample in picked]


def take(pipeline, class_name: str):
    for step in pipeline.transforms:
        if type(step).__name__ == class_name:
            return step
        if type(step).__name__ == "RandomApply" and type(step.transforms[0]).__name__ == class_name:
            return step.transforms[0]
    raise KeyError(class_name)


pipeline = book_transform()
rotation = take(pipeline, "RandomRotation")
homography = take(pipeline, "RandomPerspective")
jitter = take(pipeline, "ColorJitter")
contrast = transforms.RandomAutocontrast(p=1.0)
crop = take(pipeline, "RandomResizedCrop")
blur = take(pipeline, "GaussianBlur")
noise = take(pipeline, "GaussianNoise")
homography_only = transforms.RandomPerspective(
    distortion_scale=homography.distortion_scale,
    p=1.0,
    fill=homography.fill,
)
to_tensor = transforms.ToTensor()

print("Книги:", ", ".join(name for name, _ in book_images))
for sample in picked:
    print(sample.mark_id, sample.path)
print(
    f"Гомография distortion_scale={homography.distortion_scale}, p={homography.p}; "
    f"кроп scale={tuple(float(v) for v in crop.scale)}, ratio={tuple(float(v) for v in crop.ratio)}"
)


In [ ]:
def as_image(value) -> Image.Image:
    if isinstance(value, torch.Tensor):
        array = value.detach().clamp(0, 1).mul(255).byte().permute(1, 2, 0).numpy()
        return Image.fromarray(array)
    return value


def show_grid(rows: list[tuple[str, list]], title: str) -> None:
    columns = max(len(images) for _, images in rows)
    fig, axes = plt.subplots(
        len(rows),
        columns,
        figsize=(2.15 * columns, 2.35 * len(rows)),
        squeeze=False,
    )
    for row_index, (label, images) in enumerate(rows):
        for col_index in range(columns):
            axis = axes[row_index][col_index]
            axis.axis("off")
            if col_index >= len(images):
                continue
            axis.imshow(as_image(images[col_index]))
            if col_index == 0:
                axis.set_title(label, fontsize=9, loc="left")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()


## Поворот

`RandomRotation` на `0..180°`, заливка 114.

In [ ]:
show_grid(
    [(name, [image]) for name, image in book_images],
    "Исходный кроп",
)
show_grid(
    [(name, [image] + [rotation(image) for _ in range(4)]) for name, image in book_images],
    "Поворот 0–180°, fill 114. Первый кадр — исходник",
)


## Гомография

`RandomPerspective`, `distortion_scale 0.2`. Углы сдвигаются немного, буквы не растягиваются в ширину. В обучении вероятность 0.5.

In [ ]:
show_grid(
    [(name, [image] + [homography_only(image) for _ in range(4)]) for name, image in book_images],
    "Гомография, distortion_scale 0.2. Здесь включена всегда. Первый кадр — исходник",
)


## Яркость, контраст, автоконтраст

In [ ]:
show_grid(
    [(name, [image] + [jitter(image) for _ in range(4)]) for name, image in book_images],
    "Яркость и контраст 0.4. Первый кадр — исходник",
)
show_grid(
    [(name, [image] + [contrast(image) for _ in range(4)]) for name, image in book_images],
    "Автоконтраст. В обучении p=0.3, здесь включён всегда. Первый кадр — исходник",
)


## Квадратный кроп

`RandomResizedCrop`: масштаб `0.75..1.0`, `ratio 1`. Окно квадратное, поэтому ресайз к 224 не тянет клеймо в стороны.

In [ ]:
show_grid(
    [(name, [image] + [crop(image) for _ in range(4)]) for name, image in book_images],
    "Квадратный кроп, scale 0.75–1.0, ratio 1. Первый кадр — исходник",
)


## Размытие и шум

`GaussianBlur`, ядро 5, sigma `0.1..1.2`, в обучении вероятность 0.4. Шум `std 0.02`, вероятность 0.5.

In [ ]:
show_grid(
    [(name, [image] + [blur(image) for _ in range(4)]) for name, image in book_images],
    "GaussianBlur, ядро 5, sigma 0.1–1.2. Здесь включён всегда. Первый кадр — исходник",
)
show_grid(
    [(name, [image] + [noise(to_tensor(image)) for _ in range(4)]) for name, image in book_images],
    "Шум std 0.02. Первый кадр — исходник",
)


## Полный `book_transform`

Два вида одной пары. Гомография, цвет, автоконтраст, размытие и шум здесь срабатывают по своим вероятностям.

In [ ]:
show_grid(
    [(name, [image, pipeline(image), pipeline(image)]) for name, image in book_images],
    "Два вида одной пары. Первый кадр — исходник",
)
